In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# ==========================================
# 1. Data Preparation (Mini-Batch Gradient Descent)
# ==========================================
transform = transforms.ToTensor()

# Download the full training dataset
full_train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# Address Bias/Variance by creating a Validation Set
# We split the 60,000 training images into 50,000 for training and 10,000 for validation
train_size = 50000
val_size = 10000
train_data, val_data = random_split(full_train_data, [train_size, val_size])

# Mini-Batch Gradient Descent: We feed data in batches of 64
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# ==========================================
# 2. Model Architecture with Regularization
# ==========================================
class FashionNetReg(nn.Module):
    def __init__(self):
        super(FashionNetReg, self).__init__()
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(28 * 28, 256)
        self.relu1 = nn.ReLU()
        # REGULARIZATION: Dropout randomly turns off 30% of neurons to prevent overfitting
        self.dropout1 = nn.Dropout(0.3) 
        
        self.hidden2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        # REGULARIZATION: Dropout randomly turns off 30% of neurons
        self.dropout2 = nn.Dropout(0.2)
        
        self.output = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.hidden1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.hidden2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.output(x)
        return x

model = FashionNetReg()

# ==========================================
# 3. Setup, L2 Regularization, & Early Stopping
# ==========================================
criterion = nn.CrossEntropyLoss()

# REGULARIZATION: weight_decay=1e-4 applies L2 Regularization
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# EARLY STOPPING configurations
epochs = 50 # We set a high maximum epoch count
patience = 3 # Stop if validation loss doesn't improve for 3 epochs
best_val_loss = float('inf')
patience_counter = 0

# ==========================================
# 4. The Training Loop
# ==========================================
print("Starting training with Early Stopping...")

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train() 
    running_train_loss = 0.0
    
    for images, labels in train_loader:
        optimizer.zero_grad()
        predictions = model(images)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()
        
    avg_train_loss = running_train_loss / len(train_loader)
    
    # --- VALIDATION PHASE ---
    model.eval() # Turns off dropout!
    running_val_loss = 0.0
    
    with torch.no_grad():
        for images, labels in val_loader:
            predictions = model(images)
            loss = criterion(predictions, labels)
            running_val_loss += loss.item()
            
    avg_val_loss = running_val_loss / len(val_loader)
    
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # --- EARLY STOPPING LOGIC ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0 # Reset patience
        # Save the best model weights
        torch.save(model.state_dict(), 'best_fashion_model.pth')
    else:
        patience_counter += 1
        print(f"   -> No improvement in Val Loss. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print("   -> Early Stopping Triggered! Restoring best weights...")
            # Load the best weights back into the model
            model.load_state_dict(torch.load('best_fashion_model.pth'))
            break

# ==========================================
# 5. Evaluation & Classification Report
# ==========================================
print("\nEvaluating the best model on unseen Test Data...")
model.eval()

all_predictions = []
all_true_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted_classes = torch.max(outputs, 1)
        
        # Save predictions and true labels to numpy arrays for Scikit-Learn
        all_predictions.extend(predicted_classes.numpy())
        all_true_labels.extend(labels.numpy())

# Calculate overall accuracy
acc = accuracy_score(all_true_labels, all_predictions)
print(f"\nOverall Test Accuracy: {acc * 100:.2f}%\n")

# Generate the detailed classification report
print("Classification Report:")
print(classification_report(all_true_labels, all_predictions, target_names=classes))

Starting training with Early Stopping...
Epoch [1/50] | Train Loss: 0.6001 | Val Loss: 0.4421
Epoch [2/50] | Train Loss: 0.4309 | Val Loss: 0.4028
Epoch [3/50] | Train Loss: 0.4000 | Val Loss: 0.3509
Epoch [4/50] | Train Loss: 0.3778 | Val Loss: 0.3417
Epoch [5/50] | Train Loss: 0.3662 | Val Loss: 0.3443
   -> No improvement in Val Loss. Patience: 1/3
Epoch [6/50] | Train Loss: 0.3517 | Val Loss: 0.3244
Epoch [7/50] | Train Loss: 0.3434 | Val Loss: 0.3375
   -> No improvement in Val Loss. Patience: 1/3
Epoch [8/50] | Train Loss: 0.3342 | Val Loss: 0.3168
Epoch [9/50] | Train Loss: 0.3286 | Val Loss: 0.3154
Epoch [10/50] | Train Loss: 0.3231 | Val Loss: 0.3202
   -> No improvement in Val Loss. Patience: 1/3
Epoch [11/50] | Train Loss: 0.3158 | Val Loss: 0.3137
Epoch [12/50] | Train Loss: 0.3126 | Val Loss: 0.3129
Epoch [13/50] | Train Loss: 0.3055 | Val Loss: 0.3166
   -> No improvement in Val Loss. Patience: 1/3
Epoch [14/50] | Train Loss: 0.3027 | Val Loss: 0.2969
Epoch [15/50] | Trai

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# ==========================================
# 1. Data Preparation (Mini-Batch & Val Split)
# ==========================================
# Load the dataset
fashion_mnist = tf.keras.datasets.fashion_mnist
(x_train_full, y_train_full), (x_test, y_test) = fashion_mnist.load_data()

# Normalize pixel values to be between 0 and 1
x_train_full, x_test = x_train_full / 255.0, x_test / 255.0

# Class names for the classification report later
classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# ==========================================
# 2. Model Architecture with Regularization
# ==========================================
model = Sequential([
    Flatten(input_shape=(28, 28)),
    
    # Hidden Layer 1: 256 neurons, ReLU, L2 Regularization (Weight Decay)
    Dense(256, activation='relu', kernel_regularizer=l2(1e-4)),
    # REGULARIZATION: Dropout turns off 30% of neurons
    Dropout(0.3),
    
    # Hidden Layer 2: 128 neurons, ReLU, L2 Regularization
    Dense(128, activation='relu', kernel_regularizer=l2(1e-4)),
    Dropout(0.3),
    
    # Output Layer: 10 classes, Softmax for probabilities
    Dense(10, activation='softmax')
])

# ==========================================
# 3. Setup Optimizer and Loss
# ==========================================
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# ==========================================
# 4. Early Stopping Callback
# ==========================================
# Instead of manual tracking, Keras uses a Callback to monitor validation loss.
# restore_best_weights=True automatically reverts the model to its best state.
early_stop = EarlyStopping(monitor='val_loss', 
                           patience=3, 
                           restore_best_weights=True,
                           verbose=1)

# ==========================================
# 5. The Training Loop
# ==========================================
print("Starting training with Early Stopping...")

# validation_split=0.166 takes ~10,000 images out of the 60,000 for validation
# batch_size=64 applies our Mini-Batch Gradient Descent
history = model.fit(x_train_full, y_train_full, 
                    epochs=50, 
                    batch_size=64, 
                    validation_split=0.166, 
                    callbacks=[early_stop])

# ==========================================
# 6. Evaluation & Classification Report
# ==========================================
print("\nEvaluating the best model on unseen Test Data...")

# Get raw probability predictions for the test set
predictions_probs = model.predict(x_test)

# Convert probabilities to actual class predictions (e.g., [0.1, 0.8, 0.1] -> Class 1)
predicted_classes = np.argmax(predictions_probs, axis=1)

# Calculate overall accuracy
acc = accuracy_score(y_test, predicted_classes)
print(f"\nOverall Test Accuracy: {acc * 100:.2f}%\n")

# Generate the detailed classification report
print("Classification Report:")
print(classification_report(y_test, predicted_classes, target_names=classes))


🇬🇧 Classwork: CIFAR-10 Color Image Classification
Objective: Build a deep neural network to classify 32x32 color (RGB) images into 10 different categories using Mini-Batch Gradient Descent, Dropout, and Early Stopping.

Requirements:

Data Preparation: Load the cifar10 dataset from Keras. Scale the pixel values so they range between 0 and 1.

Model Architecture:

Add a Flatten layer. (Careful: the input shape is now (32, 32, 3)).

Create three hidden layers using Dense with the ReLU activation function. The sizes should be 512, 256, and 128 neurons, respectively.

Output layer must have 10 neurons with a softmax activation.

Regularization: Add a Dropout layer with a 20% drop rate (0.2) immediately after each of the three hidden layers to prevent overfitting.

Compilation: Use the adam optimizer and sparse_categorical_crossentropy for the loss function.

Training (Mini-Batch & Early Stopping):

Configure Early Stopping to monitor val_loss with a patience of 5 epochs. Ensure it restores the best weights.

Train the model for a maximum of 50 epochs.

Use a batch size of 128.

Set aside 20% of the training data for validation (validation_split=0.2).

Evaluation: Print the overall test accuracy and generate a classification report using Scikit-Learn.

🇦🇲 Դասարանային աշխատանք. CIFAR-10 Գունավոր պատկերների դասակարգում
Նպատակը՝ Կառուցել խորը նեյրոնային ցանց՝ 32x32 չափի գունավոր (RGB) պատկերները 10 տարբեր դասերի բաժանելու համար՝ կիրառելով մինի-փաթեթներով գրադիենտային վայրէջք (Mini-Batch Gradient Descent), Dropout ռեգուլյարիզացիա և վաղաժամ կանգառ (Early Stopping):

Պահանջները՝

Տվյալների նախապատրաստում: Ներբեռնեք cifar10 տվյալների բազան Keras-ից: Նորմալացրեք փիքսելների արժեքները այնպես, որ դրանք գտնվեն 0-ից 1 միջակայքում:

Մոդելի ճարտարապետություն:

Ավելացրեք Flatten շերտ: (Ուշադրություն՝ մուտքային տվյալների չափն այժմ (32, 32, 3) է):

Ստեղծեք երեք թաքնված շերտ (Dense)՝ ReLU ակտիվացման ֆունկցիայով: Նեյրոնների քանակը պետք է լինի համապատասխանաբար 512, 256 և 128:

Ելքային (Output) շերտը պետք է ունենա 10 նեյրոն՝ softmax ակտիվացմամբ:

Ռեգուլյարիզացիա: Յուրաքանչյուր թաքնված շերտից անմիջապես հետո ավելացրեք Dropout շերտ՝ 20% ինդեքսով (0.2), գերսովորումից (overfitting) խուսափելու համար:

Կոմպիլյացիա: Որպես օպտիմիզատոր օգտագործեք adam-ը, իսկ կորստի ֆունկցիայի համար՝ sparse_categorical_crossentropy:

Ուսուցում (Mini-Batch և Early Stopping):

Կարգավորեք վաղաժամ կանգառը (Early Stopping), որպեսզի այն հետևի val_loss-ին՝ 5 էպոխա համբերությամբ (patience=5): Համոզվեք, որ այն վերականգնում է լավագույն կշիռները:

Ուսուցանեք մոդելը առավելագույնը 50 էպոխա տևողությամբ:

Օգտագործեք 128 չափով մինի-փաթեթ (batch size = 128):

Ուսուցման տվյալների 20%-ը առանձնացրեք որպես վալիդացիայի բազմություն (validation_split=0.2):

Գնահատում: Տպեք թեստային տվյալների վերջնական ճշտությունը (accuracy) և գեներացրեք դասակարգման հաշվետվություն (classification report)՝ օգտագործելով Scikit-Learn գրադարանը:

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# 1. Data Preparation
print("Downloading CIFAR-10 dataset...")
cifar10 = tf.keras.datasets.cifar10
(x_train_full, y_train_full), (x_test, y_test) = cifar10.load_data()

# CIFAR-10 class names
classes = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 
           'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

# Scale the data (0 to 1)
x_train_full = x_train_full / 255.0
x_test = x_test / 255.0

# y_train comes as a 2D array (e.g., [[6], [9], ...]). Flatten it to 1D for sklearn later.
y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

# 2 & 3. Model Architecture & Regularization
model = Sequential([
    # Input shape is 32x32 pixels with 3 color channels (Red, Green, Blue)
    Flatten(input_shape=(32, 32, 3)),
    
    # Layer 1
    Dense(512, activation='relu'),
    Dropout(0.2),
    
    # Layer 2
    Dense(256, activation='relu'),
    Dropout(0.2),
    
    # Layer 3
    Dense(128, activation='relu'),
    Dropout(0.2),
    
    # Output Layer
    Dense(10, activation='softmax')
])

# 4. Compilation
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 5. Training Setup (Early Stopping & Mini-Batch)
early_stop = EarlyStopping(monitor='val_loss', 
                           patience=5, 
                           restore_best_weights=True,
                           verbose=0)

print("Starting training...")
history = model.fit(x_train_full, y_train_full, 
                    epochs=50, 
                    batch_size=128, # Mini-batch
                    validation_split=0.2, # 20% for validation
                    callbacks=[early_stop])

# 6. Evaluation
print("\nEvaluating the model on unseen Test Data...")
predictions_probs = model.predict(x_test)
predicted_classes = np.argmax(predictions_probs, axis=1)

# Overall Accuracy
acc = accuracy_score(y_test, predicted_classes)
print(f"\nOverall Test Accuracy: {acc * 100:.2f}%\n")

# Classification Report
print("Classification Report:")
print(classification_report(y_test, predicted_classes, target_names=classes))

/Users/rafael/projects/ml_bootcamp/.venv/lib/python3.13/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Starting training...
Epoch 1/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.2461 - loss: 2.0343 - val_accuracy: 0.3240 - val_loss: 1.8708
Epoch 2/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.3243 - loss: 1.8645 - val_accuracy: 0.3591 - val_loss: 1.7840
Epoch 3/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.3460 - loss: 1.8031 - val_accuracy: 0.3891 - val_loss: 1.7144
Epoch 4/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.3590 - loss: 1.7701 - val_accuracy: 0.4041 - val_loss: 1.6824
Epoch 5/50
174/313 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.3709 - loss: 1.7343

KeyboardInterrupt: 